# Batch Analysis with Atropos

This notebook demonstrates batch processing multiple scenarios and comparing results across different configurations.

In [ ]:
from atropos import DeploymentScenario, estimate_outcome
from atropos.presets import STRATEGIES, QUANTIZATION_BONUS
from atropos.calculations import combine_strategies
import pandas as pd
import matplotlib.pyplot as plt

## Define Multiple Scenarios

Compare different deployment sizes.

In [ ]:
# Define scenarios for different model sizes
scenarios = [
    DeploymentScenario(
        name="small-coder",
        parameters_b=7.0,
        memory_gb=5.0,
        throughput_toks_per_sec=25.0,
        power_watts=150.0,
        requests_per_day=20000,
        tokens_per_request=800,
        electricity_cost_per_kwh=0.15,
        annual_hardware_cost_usd=6000.0,
        one_time_project_cost_usd=8000.0,
    ),
    DeploymentScenario(
        name="medium-coder",
        parameters_b=34.0,
        memory_gb=14.0,
        throughput_toks_per_sec=40.0,
        power_watts=320.0,
        requests_per_day=50000,
        tokens_per_request=1200,
        electricity_cost_per_kwh=0.15,
        annual_hardware_cost_usd=24000.0,
        one_time_project_cost_usd=27000.0,
    ),
    DeploymentScenario(
        name="large-assistant",
        parameters_b=70.0,
        memory_gb=48.0,
        throughput_toks_per_sec=30.0,
        power_watts=700.0,
        requests_per_day=150000,
        tokens_per_request=2000,
        electricity_cost_per_kwh=0.12,
        annual_hardware_cost_usd=80000.0,
        one_time_project_cost_usd=60000.0,
    ),
]

print(f"Defined {len(scenarios)} scenarios")

## Batch Evaluation

In [ ]:
# Evaluate all scenarios with all strategies
results = []

for scenario in scenarios:
    for strategy_name, strategy in STRATEGIES.items():
        # Without quantization
        outcome = estimate_outcome(scenario, strategy)
        results.append({
            'Scenario': scenario.name,
            'Params (B)': scenario.parameters_b,
            'Strategy': strategy_name,
            'Quantization': False,
            'Memory (GB)': outcome.optimized_memory_gb,
            'Throughput (tok/s)': outcome.optimized_throughput_toks_per_sec,
            'Annual Savings ($)': outcome.annual_total_savings_usd,
            'Break-even (mo)': outcome.break_even_years * 12 if outcome.break_even_years else None,
            'CO2e Savings (kg/yr)': outcome.annual_co2e_savings_kg,
            'Risk': outcome.quality_risk,
        })
        
        # With quantization
        combined = combine_strategies(strategy, QUANTIZATION_BONUS)
        outcome_q = estimate_outcome(scenario, combined)
        results.append({
            'Scenario': scenario.name,
            'Params (B)': scenario.parameters_b,
            'Strategy': strategy_name,
            'Quantization': True,
            'Memory (GB)': outcome_q.optimized_memory_gb,
            'Throughput (tok/s)': outcome_q.optimized_throughput_toks_per_sec,
            'Annual Savings ($)': outcome_q.annual_total_savings_usd,
            'Break-even (mo)': outcome_q.break_even_years * 12 if outcome_q.break_even_years else None,
            'CO2e Savings (kg/yr)': outcome_q.annual_co2e_savings_kg,
            'Risk': outcome_q.quality_risk,
        })

df = pd.DataFrame(results)
df.head(10)

## Analysis & Visualization

In [ ]:
# Filter for best strategy per scenario
best_by_scenario = df.loc[df.groupby('Scenario')['Annual Savings ($)'].idxmax()]
best_by_scenario[['Scenario', 'Strategy', 'Quantization', 'Annual Savings ($)', 'Break-even (mo)']]

In [ ]:
# Plot savings by scenario
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Annual savings
pivot_savings = df.pivot_table(
    values='Annual Savings ($)', 
    index='Scenario', 
    columns=['Strategy', 'Quantization'],
    aggfunc='max'
)
pivot_savings.plot(kind='bar', ax=axes[0])
axes[0].set_title('Annual Savings by Scenario')
axes[0].set_ylabel('USD')
axes[0].tick_params(axis='x', rotation=45)

# Break-even time
pivot_breakeven = df[df['Break-even (mo)'].notna()].pivot_table(
    values='Break-even (mo)', 
    index='Scenario', 
    columns=['Strategy', 'Quantization'],
    aggfunc='min'
)
pivot_breakeven.plot(kind='bar', ax=axes[1])
axes[1].set_title('Break-even Time by Scenario')
axes[1].set_ylabel('Months')
axes[1].tick_params(axis='x', rotation=45)
axes[1].axhline(y=12, color='r', linestyle='--', label='1 year threshold')

plt.tight_layout()
plt.show()

## Export Results

In [ ]:
# Save to CSV
df.to_csv('../reports/batch_analysis.csv', index=False)
print("Saved to ../reports/batch_analysis.csv")

# Summary statistics
summary = df.groupby('Scenario').agg({
    'Annual Savings ($)': ['min', 'max', 'mean'],
    'CO2e Savings (kg/yr)': ['min', 'max'],
})
summary